# One-Hot-Encoding (OHE) nach Cleanup Skript
- Ziel: Kategoriale Features modellfähig machen (OHE + Multi-Hot)
- multi-Select-Spalten: Split per Semikolon, dann Top10 + other (optional none für Missing Werte) und Multi-Hot via MultiLabelBinarizer mit Prefix `col__`
- ordinal statt OHE: `OrgSize` und `EdLevel` werden geordnet gemappt und auf 0/1 skaliert
- restliche Objektspalten: Alles mit <= 200 unique wird one-hot encodiert (dummy_na=True), High-Cardinality bleibt unverändert
- kleine Cleanup: Alle Spalten mit "`Choice`" im Namen werden gedroppt
- abschließend wird der Datensatz als `One-Hot-Encoded-final.csv` gespeichert, welcher für anschließende Modellierungsaufgaben genutzt werden kann

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer

In [ ]:
df = pd.read_csv("survey_results_cleaned_final.csv")
max_unique = 200
na_as_category = True
TOP_N = 10

In [ ]:
multiselect_cols = [
    "LanguageHaveWorkedWith",
    "DatabaseHaveWorkedWith",
    "PlatformHaveWorkedWith",
    "WebframeHaveWorkedWith",
    "DevEnvsHaveWorkedWith",
    "OfficeStackAsyncHaveWorkedWith",
    "AIModelsHaveWorkedWith",
    "CommPlatformHaveWorkedWith",
    "AIAgent_Uses",
]
multiselect_cols = [c for c in multiselect_cols if c in df.columns]

SEP = ";"

def split_cell(x):
    if pd.isna(x) or str(x).strip() == "":
        return ["none"] if na_as_category else []
    return [p.strip() for p in str(x).split(SEP) if p.strip()]

def multiselect_topN_with_other(df: pd.DataFrame, col: str, top_n: int = 10, na_as_category: bool = True):
    s = df[col].apply(split_cell)

    exploded = s.explode()
    counts = exploded[exploded != "none"].value_counts()
    top = set(counts.head(top_n).index)

    def keep_top_and_other(items):
        if na_as_category and items == ["none"]:
            return ["none"]

        kept = [x for x in items if x in top]
        has_other = any((x not in top) and (x != "none") for x in items)

        out = set(kept)
        if has_other:
            out.add("other")
        if na_as_category and ("none" in items):
            out.add("none")
        return sorted(out)

    s2 = s.apply(keep_top_and_other)

    classes = sorted(top) + ["other"]
    if na_as_category:
        classes = ["none"] + classes

    mlb = MultiLabelBinarizer(classes=classes)
    dummies = pd.DataFrame(mlb.fit_transform(s2), columns=mlb.classes_, index=df.index)

    dummies = dummies.add_prefix(f"{col}__")

    return dummies, sorted(top)

# Multi-Select auf Top10+other
for col in multiselect_cols:
    dummies, top10 = multiselect_topN_with_other(df, col, top_n=TOP_N, na_as_category=na_as_category)
    df = df.drop(columns=[col]).join(dummies)

    print(f"Top{TOP_N} für {col}: {top10}")

print("Form nach Multi-Select-OHE (Top10+other):", df.shape)
print("Ursprüngliche DataFrame-Form:", df.shape)

In [ ]:
orgsize_order = [
    "just me - i am a freelancer, sole proprietor, etc.",
    "less than 20 employees",
    "20 to 99 employees",
    "100 to 499 employees",
    "500 to 999 employees",
    "1,000 to 4,999 employees",
    "5,000 to 9,999 employees",
    "10,000 or more employees",
]
orgsize_unknown = {"i don’t know"}

if "OrgSize" in df.columns:
    orgsize_map = {cat: i for i, cat in enumerate(orgsize_order)}
    max_val = len(orgsize_order) - 1

    df["OrgSize"] = (
        df["OrgSize"]
        .replace(list(orgsize_unknown), np.nan)
        .map(orgsize_map)
        .astype(float)
        / max_val
    )

edlevel_order = [
    "primary/elementary school",
    "secondary school",
    "some college/university study without earning a degree",
    "associate degree",
    "bachelor’s degree",
    "master’s degree",
    "professional degree",
]
edlevel_unknown = {"other"}

if "EdLevel" in df.columns:
    edlevel_map = {cat: i for i, cat in enumerate(edlevel_order)}
    max_val = len(edlevel_order) - 1

    df["EdLevel"] = (
        df["EdLevel"]
        .replace(list(edlevel_unknown), np.nan)
        .map(edlevel_map)
        .astype(float)
        / max_val
    )

for col, unknown_set in [("OrgSize", orgsize_unknown), ("EdLevel", edlevel_unknown)]:
    if col in df.columns:
        unmapped = (
            df.loc[df[col].isna(), col]  # ist nach Mapping NaN
        )

In [ ]:
object_columns = df.select_dtypes(include=["object"]).columns.tolist()

print("Object-Spalten:")
object_columns

In [ ]:
if object_columns:
    nunique_per_column = df[object_columns].nunique(dropna=True)
else:
    nunique_per_column = pd.Series(dtype=int)

In [ ]:
columns_to_encode = nunique_per_column[nunique_per_column <= max_unique].index.tolist()

high_cardinality_columns = nunique_per_column[nunique_per_column > max_unique].index.tolist()

print("Spalten für One-Hot-Encoding (≤ 50 Werte):")
print(columns_to_encode)

print("\nSpalten mit hoher Kardinalität (> 50 Werte):")
high_cardinality_columns

# EducationLevel und OrgSize wurden oben ordinal encodiert -> nicht one-hot encoden
exclude_cols = [c for c in ['EdLevel','OrgSize'] if c in df.columns]
columns_to_encode = [c for c in columns_to_encode if c not in exclude_cols]
high_cardinality_columns = [c for c in high_cardinality_columns if c not in exclude_cols]

In [ ]:
df_encoded = pd.get_dummies(
    df,
    columns=columns_to_encode,
    dummy_na=na_as_category,
    drop_first=False
)

print("Neue DataFrame-Form nach One-Hot-Encoding:", df_encoded.shape)

choice_cols = [c for c in df_encoded.columns if "Choice" in c]

print(f"Dropping Choice columns: {len(choice_cols)}")

print("Examples:", choice_cols[:10])

df_encoded = df_encoded.drop(columns=choice_cols)

print("Form nach Drop Choice:", df_encoded.shape)

In [ ]:
if high_cardinality_columns:
    print("Nicht encodierte Spalten mit mehr als", max_unique, "verschiedenen Einträgen:")
    for col in high_cardinality_columns:
        print(f" - {col}: {int(nunique_per_column[col])} unique values")
else:
    print("Keine Spalten mit hoher Kardinalität gefunden.")

In [ ]:
df_encoded.to_csv("One-Hot-Encoded-final.csv", index=False)
print("Shape: ", df_encoded.shape)
print("One-Hot-Encoded-final.csv wurde gespeichert")